<a href="https://colab.research.google.com/github/loukikjoshi06-ops/SIH-26016-App/blob/main/model_test_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# 1. Check GPU
!nvidia-smi

# 2. Install required AI libraries
!pip install -q transformers bitsandbytes accelerate peft qwen-vl-utils

# 3. Clone your GitHub repository to get the dataset
!git clone https://github.com/loukikjoshi06-ops/SIH-26016-App.git || echo "Repo already downloaded"
print("✅ Libraries installed and dataset downloaded!")

Mon Sep 28 17:47:07 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   35C    P8             11W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import torch
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig

model_id = "Qwen/Qwen2.5-VL-7B-Instruct"

# 4-bit config to save memory
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    bnb_4bit_use_double_quant=True
)

processor = AutoProcessor.from_pretrained(model_id, min_pixels=256*28*28, max_pixels=1280*28*28)

print("⏳ Downloading and loading the 7B Model (This takes ~2 minutes)...")
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    quantization_config=bnb_config
)
print("✅ Model loaded successfully!")

preprocessor_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/1.05k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.37k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/5.70k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/216 [00:00<?, ?B/s]

✅ Model loaded successfully!


In [3]:
import os, json, re, torch
from PIL import Image
from qwen_vl_utils import process_vision_info

# Helper functions to convert Marathi numbers to English
def marathi_to_english_digits(text):
    if not isinstance(text, str): return text
    return text.translate(str.maketrans("०१२३४५६७८९", "0123456789"))

def standardize_json(data):
    if isinstance(data, dict): return {k: standardize_json(v) for k, v in data.items()}
    elif isinstance(data, list): return [standardize_json(item) for item in data]
    elif isinstance(data, str): return marathi_to_english_digits(data)
    return data

# The perfect prompt we created earlier
SYSTEM_PROMPT = """You are an official Maharashtra Land Records (भूमी अभिलेख) extraction system.
Extract all details from this 7/12 (सातबारा) document into a valid JSON object.
RULES: Keep text in original Marathi Devanagari. Read numerals EXACTLY as they appear.

SCHEMA:
{"district": "जिल्हा", "taluka": "तालुका", "village": "गाव (word before the 6-digit code)", "village_code": "६ अंकी कोड", "survey_number": "गट क्र.", "khata_number": "खाते क्र.", "occupancy_class": "भोगवटदार वर्ग", "owners": ["नाव १", "नाव २"], "area": {"unit": "हे.आर.चौ.मी", "total": "एकूण क्षेत्र", "cultivable": "लागवडी योग्य", "uncultivable": "पोटखराब"}, "assessment": "आकारणी", "encumbrances": "इतर अधिकार / बोजा (null if empty)", "last_mutation_number": "शेवटचा फेरफार क्रमांक"}"""

# Folder where your dataset is stored
data_folder = "SIH-26016-App/App/AI/data/processed/7-12"
image_files = [f for f in os.listdir(data_folder) if f.endswith(('.png', '.jpg', '.jpeg'))]

print(f"📂 Found {len(image_files)} images to test!")
all_results = {}

# Loop through every image and test the AI
for img_name in image_files:
    print(f"\n⏳ Processing: {img_name}")
    img_path = os.path.join(data_folder, img_name)
    raw_img = Image.open(img_path).convert("RGB")

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": [{"type": "image", "image": raw_img}, {"type": "text", "text": "Extract all data into the requested JSON schema."}]}
    ]

    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    vision_imgs, _ = process_vision_info(messages)
    inputs = processor(text=[text], images=vision_imgs, return_tensors="pt", padding=True, truncation=False).to(model.device)

    with torch.inference_mode():
        output_ids = model.generate(**inputs, max_new_tokens=768, temperature=0.01)

    raw_output = processor.decode(output_ids[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    clean_text = raw_output.replace("```json", "").replace("```", "").strip()

    # Parse and repair JSON safely
    match = re.search(r'\{.*\}', clean_text, re.DOTALL)
    json_str = match.group(0) if match else clean_text
    json_str = re.sub(r'([0-9"a-zA-Z\u0900-\u097F])\s*\n\s*(")', r'\1,\n\2', json_str)
    json_str = re.sub(r',\s*([\}\]])', r'\1', json_str)

    try:
        extracted_json = json.loads(json_str)
    except:
        extracted_json = {"error": "Failed to parse", "raw": raw_output}

    # Convert numbers to English and save
    final_json = standardize_json(extracted_json)
    all_results[img_name] = final_json
    print(f"✅ Finished! Extracted District: {final_json.get('district', 'N/A')}")

# Save the final report
with open("test_results.json", "w", encoding="utf-8") as f:
    json.dump(all_results, f, indent=2, ensure_ascii=False)

print("\n🎉 ALL DONE! Open the 'test_results.json' file in your Colab files tab to see the accuracy report.")

📂 Found 18 images to test!

⏳ Processing: Gemini_Generated_Image_pegaf5pegaf5pega.png
✅ Finished! Extracted District: भारणे

⏳ Processing: Gemini_Generated_Image_96i1bc96i1bc96i1.png
✅ Finished! Extracted District: पुणे

⏳ Processing: 7-12-sample-of-Maharashtra_page-0002.jpg
✅ Finished! Extracted District: पुणे

⏳ Processing: ChatGPT Image Sep 27, 2026, 09_17_40 PM.png
✅ Finished! Extracted District: नाशिक

⏳ Processing: Gemini_Generated_Image_3tzufo3tzufo3tzu.png
✅ Finished! Extracted District: N/A

⏳ Processing: Gemini_Generated_Image_jw0eecjw0eecjw0e.png
✅ Finished! Extracted District: भोगवद्वार

⏳ Processing: Gemini_Generated_Image_siypqvsiypqvsiyp (1).png
✅ Finished! Extracted District: पुणे

⏳ Processing: Gemini_Generated_Image_ler5dwler5dwler5.png
✅ Finished! Extracted District: भोगद्वार

⏳ Processing: Gemini_Generated_Image_wr6wb6wr6wb6wr6w.png
✅ Finished! Extracted District: सारी

⏳ Processing: Gemini_Generated_Image_nklcvqnklcvqnklc.png
✅ Finished! Extracted District: भोगला ब